# 00 — Environment and performance strategy

This notebook verifies the Python environment and establishes the rule that will keep the reconstructed project fast:

> **Use SymPy to derive expressions once. Never perform symbolic substitution, differentiation, or simplification inside the geodesic integration loop.**

Our numerical path will be:

`SymPy -> common-subexpression elimination -> lambdify -> NumPy -> optional Numba JIT -> RK4 / solve_ivp`


In [ ]:
import sys
import numpy as np
import scipy
import sympy as sp
import matplotlib
import numba

print('Python     :', sys.version.split()[0])
print('NumPy      :', np.__version__)
print('SciPy      :', scipy.__version__)
print('SymPy      :', sp.__version__)
print('Matplotlib :', matplotlib.__version__)
print('Numba      :', numba.__version__)


## Why the old approach became painfully slow

Calling operations such as `expr.subs(...)`, `sp.diff(...)`, or broad `sp.simplify(...)` repeatedly during RK4 makes each RHS evaluation symbolic. RK4 needs thousands to millions of RHS evaluations, so that design can become effectively unusable.

Instead, we turn a symbolic expression into an ordinary numerical function once.


In [ ]:
# Small demonstration: symbolic expression -> fast numerical function
x, y, z, a = sp.symbols('x y z a', real=True)

expr = sp.sqrt((x**2 + y**2 + z**2 - a**2)**2 + 4*a**2*z**2)

# cse=True asks SymPy to identify repeated algebraic pieces before numerical evaluation.
# docstring_limit=0 avoids spending time rendering enormous generated expressions.
expr_num = sp.lambdify(
    (x, y, z, a),
    expr,
    modules='numpy',
    cse=True,
    docstring_limit=0,
)

expr_num(10.0, 2.0, 1.0, 0.5)


## Optional second acceleration layer: Numba

If the final Kerr geodesic RHS is still expensive after `lambdify`, we will generate a scalar numerical Python function and JIT-compile it with Numba. This keeps the scientific source in Python while compiling the hot numerical loop to machine code.


In [ ]:
from numba import njit

@njit(cache=False)
def rk4_step(rhs, lam, state, h, M, spin):
    k1 = rhs(lam, state, M, spin)
    k2 = rhs(lam + 0.5*h, state + 0.5*h*k1, M, spin)
    k3 = rhs(lam + 0.5*h, state + 0.5*h*k2, M, spin)
    k4 = rhs(lam + h, state + h*k3, M, spin)
    return state + (h/6.0)*(k1 + 2.0*k2 + 2.0*k3 + k4)

print('RK4 template defined. We will connect it to the generated geodesic RHS later.')


## Rules for the reconstruction

1. Use geometrized units initially: $G=c=1$.
2. Use a single, explicitly documented metric-signature convention.
3. Do **not** use `Matrix.inv()` for Kerr-Schild if the analytic inverse is available.
4. Do **not** apply `simplify()` indiscriminately to huge tensor expressions.
5. Exploit stationarity: $\partial_t g_{\mu\nu}=0$.
6. Compute only quantities needed by the contracted geodesic RHS.
7. Perform CSE jointly on the final RHS components so repeated expressions are shared.
8. Cache expensive symbolic derivations / generated numerical source.
9. Validate $M=0$, $a=0$, four-velocity/null normalization, and integrator convergence before trusting Kerr trajectories.

The next notebook will construct and verify the Kerr-Schild geometry carefully before any orbit integration is attempted.
